In [1]:
# Import the libraries

%pip install yfinance 
%pip install plotly
%pip install scikit-learn

import yfinance as yf

import pandas as pd 
import numpy as np 

import plotly.graph_objects as go 
import plotly.express as px 
#import plotly.io as pio



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Download the stock data from yfinance
import yfinance as yf
# Define the stock ticker and the date range
tickers = ['AAPL']
#tickers.append('GC=F')  # Gold futures
#tickers.append('BTC-USD')  # Bitcoin in USD
start_date = '2020-01-01'
end_date = '2026-01-01'

# Download the stock data
stocks = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)
gold = yf.download('GC=F', start=start_date, end=end_date, auto_adjust=True)
BTC = yf.download('BTC-USD', start=start_date, end=end_date, auto_adjust=True)
sp500 = yf.download('^GSPC', start=start_date, end=end_date, auto_adjust=True)
nasdaq = yf.download('^IXIC', start=start_date, end=end_date, auto_adjust=True)
dowjones = yf.download('^DJI', start=start_date, end=end_date, auto_adjust=True)   

# Arrange the data so that 'Close' price can directly be used for plotting and calculations
stocks.columns = stocks.columns.droplevel(1)  # Drop the ticker level from columns
gold.columns = gold.columns.droplevel(1)
BTC.columns = BTC.columns.droplevel(1)
sp500.columns = sp500.columns.droplevel(1)
nasdaq.columns = nasdaq.columns.droplevel(1)
dowjones.columns = dowjones.columns.droplevel(1)

# Add the major indices as features to the stocks DataFrame
stocks['sp500'] = sp500['Close']
stocks['nasdaq'] = nasdaq['Close']
stocks['dowjones'] = dowjones['Close']

# reset the index to make 'Date' a column
#stocks.reset_index(inplace=True)
# Drop the Open, High, Low columns
stocks.drop(columns=['Open', 'High', 'Low'], inplace=True)

stocks.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Price,Close,Volume,sp500,nasdaq,dowjones
Date,,,,,
2020-01-02,72.400536,135480400,3257.850098,9092.190430,28868.800781
2020-01-03,71.696625,146322800,3234.850098,9020.769531,28634.880859
2020-01-06,72.267929,118387200,3246.280029,9071.469727,28703.380859
2020-01-07,71.928040,108872000,3237.179932,9068.580078,28583.679688
2020-01-08,73.085129,132079200,3253.050049,9129.240234,28745.089844


In [ ]:
# Normalize the gold and bitcoin prices to the first value to compare relative changes
def normalize_to_first(df):
    return df / df.iloc[0]

stocks['Gold'] = normalize_to_first(gold['Close'])
stocks['Bitcoin'] = normalize_to_first(BTC['Close'])


# Add the target variable, the direction of the stock price movement the next day
stocks['Target'] = np.where(stocks['Close'].shift(-1) > stocks['Close'], 1, 0)

# Add a target variable which predicts that over next 5 days, the stock price will go up at least once
stocks.head(20)

Price,Close,Volume,sp500,nasdaq,dowjones,Gold,Bitcoin,Target
Date,,,,,,,,
2020-01-02,72.400536,135480400,3257.850098,9092.190430,28868.800781,1.000000,0.970181,0
2020-01-03,71.696625,146322800,3234.850098,9020.769531,28634.880859,1.016202,1.020098,1
2020-01-06,72.267929,118387200,3246.280029,9071.469727,28703.380859,1.027353,1.079032,0
2020-01-07,71.928040,108872000,3237.179932,9068.580078,28583.679688,1.031027,1.133819,1
2020-01-08,73.085129,132079200,3253.050049,9129.240234,28745.089844,1.021581,1.122176,1
2020-01-09,74.637489,170108400,3274.699951,9203.429688,28956.900391,1.017842,1.094289,1
2020-01-10,74.806221,140644800,3265.350098,9178.860352,28823.769531,1.021646,1.134216,1
2020-01-13,76.404411,121532000,3288.129883,9273.929688,28907.050781,1.015677,1.131111,0
2020-01-14,75.372704,161954400,3283.149902,9251.330078,28939.669922,1.011742,1.226049,0


In [4]:
### Add Bollinger Bands to the stock data
# Calculate the 20-day moving average and standard deviation


def Bollinger_Bands(df, window=20, num_std_dev=2):
    df['MA20'] = df['Close'].rolling(window=window).mean()
    df['STD20'] = df['Close'].rolling(window=window).std()
    df['UpperBand'] = df['MA20'] + (df['STD20'] * num_std_dev)
    df['LowerBand'] = df['MA20'] - (df['STD20'] * num_std_dev)
    return df
stocks = Bollinger_Bands(stocks)  
#stocks.dropna(inplace=True)
#.reset_index(inplace=True)  # Drop rows with NaN values resulting from rolling calculations

stocks.head()

# Plot the stock price and Bollinger Bands
#pio.renderers.default = "browser"
def plot_bollinger_bands(stocks):
    fig = go.Figure().update_layout(width=1000, height=600, title='AAPL Stock Price with Bollinger Bands', xaxis_title='Date', yaxis_title='Price (USD)')
    fig.add_trace(go.Scatter(x=stocks.index, y=stocks['Close'], mode='lines', name='Close Price'))

    fig.add_trace(go.Scatter(x=stocks.index, 
                            y=stocks['UpperBand'], 
                            mode = 'lines', 
                            name='Upper Bollinger Band', 
                            line=dict(color='red', dash='dot')))

    fig.add_trace(go.Scatter(x=stocks.index, 
                            y=stocks['LowerBand'], 
                            mode = 'lines', 
                            name='Lower Bollinger Band', 
                            line=dict(color='red', dash='dot')))
    fig.show()


In [5]:
# Get 20 day, 50 day, and 200 day moving averages for price
stocks['MA50'] = stocks['Close'].rolling(window=50).mean()
stocks['MA200'] = stocks['Close'].rolling(window=200).mean()

# Get the lag features for the closing price
stocks['Lag1'] = stocks['Close'].shift(1)
stocks['Lag2'] = stocks['Close'].shift(2)

# Moving average of volume for last 5 days
stocks['Volatility'] = stocks['Close'].rolling(window=5).std()

# Capture volume spikes
stocks['VolumeSpike'] = np.where(stocks['Volume'] > 
                                 (stocks['Volume'].rolling(window=20).mean() + 
                                  2*stocks['Volume'].rolling(window=20).std()), 1, 0)

# 
def RSI(df, period=14):
    delta = df['Close'].diff()
    gains = delta.where(delta > 0, 0)
    losses = -delta.where(delta < 0, 0)
    avg_gains = gains.rolling(window=period).mean()
    avg_losses = losses.rolling(window=period).mean()
    rs = avg_gains/avg_losses
    #print(rs)
    df['RSI'] = 100 - (100/(1 + rs))
    return df

RSI(stocks, 14)

# MACD - Moving Average Convergence Divergence
def MACD(df, short_window=12, long_window=26, signal_window=9):
    df['EMA12'] = df['Close'].ewm(span=short_window).mean()
    df['EMA26'] = df['Close'].ewm(span=long_window).mean()
    df['MACD'] = df['EMA12'] - df['EMA26']
    df['SignalLine'] = df['MACD'].ewm(span=signal_window).mean()
    df['MACD_Histogram'] = df['MACD'] - df['SignalLine']
    return df

MACD(stocks)
###stocks.dropna(inplace=True)
stocks.head()


Price,Close,Volume,sp500,nasdaq,dowjones,Gold,Bitcoin,Target,MA20,STD20,...,Lag1,Lag2,Volatility,VolumeSpike,RSI,EMA12,EMA26,MACD,SignalLine,MACD_Histogram
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,72.400536,135480400,3257.850098,9092.190430,28868.800781,1.000000,0.970181,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,72.400536,72.400536,0.000000,0.000000,0.000000
2020-01-03,71.696625,146322800,3234.850098,9020.769531,28634.880859,1.016202,1.020098,1,NaN,NaN,...,72.400536,NaN,NaN,0,NaN,72.019251,72.035043,-0.015793,-0.008774,-0.007019
2020-01-06,72.267929,118387200,3246.280029,9071.469727,28703.380859,1.027353,1.079032,0,NaN,NaN,...,71.696625,72.400536,NaN,0,NaN,72.116310,72.118717,-0.002407,-0.006165,0.003757
2020-01-07,71.928040,108872000,3237.179932,9068.580078,28583.679688,1.031027,1.133819,1,NaN,NaN,...,72.267929,71.696625,NaN,0,NaN,72.056880,72.065412,-0.008532,-0.006966,-0.001565
2020-01-08,73.085129,132079200,3253.050049,9129.240234,28745.089844,1.021581,1.122176,1,NaN,NaN,...,71.928040,72.267929,0.530805,0,NaN,72.336252,72.301888,0.034363,0.005328,0.029035


In [6]:
stocks['VolumeSpike'].value_counts()


stocks.head()

Price,Close,Volume,sp500,nasdaq,dowjones,Gold,Bitcoin,Target,MA20,STD20,...,Lag1,Lag2,Volatility,VolumeSpike,RSI,EMA12,EMA26,MACD,SignalLine,MACD_Histogram
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,72.400536,135480400,3257.850098,9092.190430,28868.800781,1.000000,0.970181,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,72.400536,72.400536,0.000000,0.000000,0.000000
2020-01-03,71.696625,146322800,3234.850098,9020.769531,28634.880859,1.016202,1.020098,1,NaN,NaN,...,72.400536,NaN,NaN,0,NaN,72.019251,72.035043,-0.015793,-0.008774,-0.007019
2020-01-06,72.267929,118387200,3246.280029,9071.469727,28703.380859,1.027353,1.079032,0,NaN,NaN,...,71.696625,72.400536,NaN,0,NaN,72.116310,72.118717,-0.002407,-0.006165,0.003757
2020-01-07,71.928040,108872000,3237.179932,9068.580078,28583.679688,1.031027,1.133819,1,NaN,NaN,...,72.267929,71.696625,NaN,0,NaN,72.056880,72.065412,-0.008532,-0.006966,-0.001565
2020-01-08,73.085129,132079200,3253.050049,9129.240234,28745.089844,1.021581,1.122176,1,NaN,NaN,...,71.928040,72.267929,0.530805,0,NaN,72.336252,72.301888,0.034363,0.005328,0.029035


In [7]:
# Add the seasonality features
stocks['DayOfWeek'] = stocks.index.dayofweek
stocks['Month'] = stocks.index.month
stocks['Quarter'] = stocks.index.quarter

stocks.head()

Price,Close,Volume,sp500,nasdaq,dowjones,Gold,Bitcoin,Target,MA20,STD20,...,VolumeSpike,RSI,EMA12,EMA26,MACD,SignalLine,MACD_Histogram,DayOfWeek,Month,Quarter
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,72.400536,135480400,3257.850098,9092.190430,28868.800781,1.000000,0.970181,0,NaN,NaN,...,0,NaN,72.400536,72.400536,0.000000,0.000000,0.000000,3,1,1
2020-01-03,71.696625,146322800,3234.850098,9020.769531,28634.880859,1.016202,1.020098,1,NaN,NaN,...,0,NaN,72.019251,72.035043,-0.015793,-0.008774,-0.007019,4,1,1
2020-01-06,72.267929,118387200,3246.280029,9071.469727,28703.380859,1.027353,1.079032,0,NaN,NaN,...,0,NaN,72.116310,72.118717,-0.002407,-0.006165,0.003757,0,1,1
2020-01-07,71.928040,108872000,3237.179932,9068.580078,28583.679688,1.031027,1.133819,1,NaN,NaN,...,0,NaN,72.056880,72.065412,-0.008532,-0.006966,-0.001565,1,1,1
2020-01-08,73.085129,132079200,3253.050049,9129.240234,28745.089844,1.021581,1.122176,1,NaN,NaN,...,0,NaN,72.336252,72.301888,0.034363,0.005328,0.029035,2,1,1


In [8]:
# Drop rows with NaN values resulting from rolling calculations
stocks.dropna(inplace=True)

# Remove the date column as it won't be used for modeling
stocks.reset_index(inplace=True)
# Remove the features that won't be used for modeling
stocks.drop(columns=['Date', 'EMA12', 'EMA26', 'STD20', 'MACD', 'SignalLine'], inplace=True)
stocks.reset_index(inplace=True)
#stocks.drop(columns=['price'], inplace=True)

stocks.head()

Price,index,Close,Volume,sp500,nasdaq,dowjones,Gold,Bitcoin,Target,MA20,...,MA200,Lag1,Lag2,Volatility,VolumeSpike,RSI,MACD_Histogram,DayOfWeek,Month,Quarter
0,0,117.193405,112559200,3483.340088,11713.870117,28494.199219,1.248409,1.596538,0,111.411876,...,85.802222,117.659409,117.572014,2.561860,0,63.102287,0.872263,3,10,4
1,1,115.552635,115393800,3483.810059,11671.559570,28606.310547,1.246835,1.572479,0,112.003135,...,86.017982,117.193405,117.659409,1.893022,0,56.510591,0.660105,4,10,4
2,2,112.601196,120639300,3426.919922,11478.879883,28195.419922,1.250508,1.630799,1,112.289541,...,86.222505,115.552635,117.193405,2.140918,0,52.833592,0.295624,0,10,4
3,3,114.086617,124423700,3443.120117,11516.490234,28308.789062,1.253132,1.655006,0,112.566238,...,86.431599,112.601196,115.552635,2.115483,0,52.563341,0.137252,1,10,4
4,4,113.465263,89946000,3435.560059,11484.690430,28210.820312,1.262447,1.781025,0,113.039536,...,86.639285,114.086617,112.601196,1.815051,0,50.121909,-0.018394,2,10,4


# Modeling - using Logistical Regression

In [15]:
# Test train split

X = stocks.drop(columns=['Target'])
y = stocks['Target']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)


from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

score = lr.score(X_test, y_test)

print(f"Logistic Regression Accuracy: {score:.2f}")


Logistic Regression Accuracy: 0.53


## Using other classification models like RandomForest, KNN

In [10]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)    

score = rf.score(X_test, y_test)
print(f"Random Forest Accuracy: {score:.2f}")

Random Forest Accuracy: 0.52


In [11]:
# KNN

from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

score = knn.score(X_test, y_test)
print(f"KNN Accuracy: {score:.2f}")

KNN Accuracy: 0.53


### Choose the Model hyperparameters using GridSearchCV

In [16]:

from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1],
    'fit_intercept': [True, False]
}

grid_search = GridSearchCV(lr, param_grid)
grid_search.fit(X_train, y_train)
print(f"Best Hyperparameters: {grid_search.best_params_}")

Best Hyperparameters: {'C': 0.1, 'fit_intercept': False}


### Using models with updated parameters

In [17]:
lr_tuned = LogisticRegression(C=grid_search.best_params_['C'], 
    fit_intercept=grid_search.best_params_['fit_intercept'], max_iter=1000, random_state=42)
lr_tuned.fit(X_train, y_train)
score = lr_tuned.score(X_test, y_test)
print(f"Tuned Logistic Regression Accuracy: {score:.2f}")

Tuned Logistic Regression Accuracy: 0.53


In [13]:
def AddFEDFeatures(stocks, start_date, end_date):
    # Use the FED data to get the following features:
    # - Federal Funds Rate
    # - Inflation Rate
    # - Unemployment Rate
    %pip install fredapi
    from fredapi import Fred
    fred = Fred(api_key='c06a7676bffb4c8338823496ee6287bf')

    interest_rate = fred.get_series('FEDFUNDS', observation_start=start_date, observation_end=end_date)
    inflation_rate = fred.get_series('CPIAUCSL', observation_start=start_date, observation_end=end_date)
    unemployment_rate = fred.get_series('UNRATE', observation_start=start_date, observation_end=end_date)

    print(interest_rate.head())

    stocks['FederalFundsRate'] = interest_rate
    stocks['InflationRate'] = inflation_rate
    stocks['UnemploymentRate'] = unemployment_rate

